# PA2 — complete Kaggle run

This notebook runs the required baseline, SFT, two from-scratch DPO tracks, persona LoRA-DPO, fixed-set generation, blinded pair judging, plots, and a PDF report. It clones your GitHub fork and applies five included source files: progress metrics, persona-style judging, plotting, the persona JSON fix, and a judge that retries individual cases after a failed batch.

**Before the overnight run:** choose **GPU T4 × 2** and turn **Internet on** in Kaggle. Run the setup and preflight cells once. They must show two GPUs, 7 passing adapter checks, and a working Kaggle judge model. Then use **Save Version → Save & Run All**. That creates a clean background run and saves `/kaggle/working` output when it completes. A draft Run All and a plain Quick Save do not provide the same output guarantee. The assignment estimates about 6.5–8.5 hours; Kaggle allows up to 12 hours for a saved run.

No training dataset upload is needed. The notebook downloads UltraFeedback, HH-RLHF, and AlpacaEval from Hugging Face. The exact evaluation subsets, model checkpoints, metrics, generations, pair files, judgments, plots, report PDF, and compact evidence ZIP are saved under `/kaggle/working/outputs` or `/kaggle/working/PA2_evidence.zip`. The report is generated from the actual run and should be reviewed before submission. A final checklist flags missing or failed stages.

The judge uses Kaggle Benchmarks model access. It refreshes a short-lived proxy credential before each judging stage. Credentials stay in `/tmp`, outside saved outputs. Kaggle model availability and quota can change, so the notebook reports unresolved judgments rather than inventing results.


In [ ]:
from pathlib import Path
import base64, importlib.metadata, json, os, subprocess, sys, zlib

REPO_URL = "https://github.com/Hoax01/small-model-alignment-starter.git"
REPO = Path("/kaggle/working/small-model-alignment-starter")
OUT = Path("/kaggle/working/outputs")
OUT.mkdir(parents=True, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not REPO.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
assert (REPO / "submission_adapter.py").is_file(), "Fork clone is missing the PA2 implementation"

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)

# Keep Kaggle's PyTorch/CUDA build. A fresh subprocess sees the protobuf upgrade.
pip("install", "-q", "-r", str(REPO / "requirements.txt"))
pip("install", "-q", "-e", str(REPO), "--no-deps")
pip("install", "-q", "kaggle_benchmarks", "protobuf>=5.29.6,<6", "python-dotenv", "reportlab", "pypdf")
try:
    from packaging.version import Version
    if Version(importlib.metadata.version("torchao")) < Version("0.16.0"):
        pip("uninstall", "-y", "torchao")
except importlib.metadata.PackageNotFoundError:
    pass

# These exact fixes travel inside the notebook, even before your fork is updated.
PATCHED_FILES = {'alignment/training.py': 'eNrtWlmP3MYRfp9f0SFghLS49M4qEoKBRkgQx35xEkESkofJgOgle3bo5WU2qdUq0X9PVfXB5jHHyjpiwwIk7XRXV3fdVd+s53mvCp7nIRNv6zxLspa1Dc/KrLxheVXVku2qhrV7wbiU2U1ZiLL9vWSvvnvNeJmyb1/8A+mTWxl5nrdY7JqqYHG869quEXHMsqKumhZIy6rlbVaVcrHQaz/KqjQ/F7zdm58b4FsV5lObFUJxTXnLkxxeIaRlK9MsacN+S1HWwC3Prg3VC2Ru+VVNsldk9GPUtVkuI+Rg6L+Fn3+oeCoaLc9gF9QBr41r2KhbvV9d/yiSNnvTv8xfMPgTN0LWILOI8+oGTlzLGA/E17xN9iGRXHdZnsZpXU0X5a51FxO4sGsF0eaVlPOr6gK8bkSA3NSxYLFY/KlXGf2LBv1LVe6ymxWdqroWTsVp1qyYbBtaS8WbLBH0ma2Zl3QpX116tFXwt3Euypt2v2JZ2cL2k+WVEgUFiGX2TpgdtX7T8DQDX4p5knRFl5NzxLIVtTSEy6dEmQveoDvGDW+ByS6vOLERF09o/05kN3t4qkj4fb99GS3VLm+KrsazWeXuXj6mbVFXyb6/0QqDz4tLMHV/Zhld0vauXj5dseuqymHxddMJWpVCpIbNH67mNAyhckrDdQUReB8fUXQjdqIRZSJmiZanrIE7ynNHBFdPnh4w1/Jccz2+Omaupbh4etRcSrnXouVTI340K1U15BMQrrF64ykv7rx5E6Zix2Je1L7RtcoY6hMkzJJf53gEbwmUVbOdDpOova8FW2vLeJQszQGiVNaENFlqtrxrq4TLVt8WI4O1Og0Zjj4pQhJ1+TRQcmICSqqyFW9bJ+eVXZ7r1YVzk7PsB0ZCmfBcNA8Qcvjsoo6+Bzu8UlzMg/W59Ult9M8A9fvWBgG7eM7+XpVCX0lFIbI0Snj1goKXHc/j0R5YQm3jjVEmY/6GZzle6Qe9BRwSh00MBVGzMl5Q16JMY6xZuY/1ZUVlJWRNdbdiWIVGD77L2j0VoqiCk76ndJJUKYTG2uva3cUfvQAKGPqh4EX/IvU5umuyVvh4X5R2RS19uAg5SKyrXCZZtv6O51IE7BHz/l16wZjBLu/k3jEyfyP8okoFVPq2uhUlRkHIXFFkSxGLwrD/kigQCPjfSDSSqriFzAWagGTUyjUGGHYQmWzj6pY+qgfRjRFeDnlHUGcBVqrVfXwHZUk0Gc+zd5RQnIP2jbOHrYnpzSyTDPoL54n4hwjZN8wz/UxMxBHq1AuUgmOKBEfLRBKCA6Yg1/oqmDGa0ajOo5Qx0TpyGB2bQaOABBtP/extA2qqYAkuwv/k1jDFMj5hPLDYfArv089knz2DROrkHJ5Jwf7J8078tWkqiNjpiaKTLaRiqEcyw7ZGe5cikuAVM8IrqzVdmYAGU6DZbFWCAkkVNQqrWbgBCJLFWYpce5uT0oWvqEPGU+iGapGAp8REJLXzWzYgOLzdt9wC9nxGU5a8l2ZwLRQlvNZy2awmPLbgt7dZPX5O77gDNUQqb2hBgkH2NDR9Gt6LtMMcaqsUGr7FnGTrbDjX0Gjjqx0QCOh85yD7GlXhQ4kdng7AlVVft2M7nkAq9JHeyY6gVc0UEzZusmd6ZahLLROdh4S0DCDuFN3CUfcNtMNo5yIr/SW+xj5LHbzQZ/A0bi0H8vfbQTCuoJfRE5ASmcLtOExESSV9+qHOYMdcbkQeVDBSd5Q3vQWiH3hxnfIfXrqmUCqi5lmdKytqQCDHogbFGwgprhrtmUSr4jhRDWDfbVNqJSOunB6bnLKvwr46pj8F2tgytoPaWudZs+CkXnyWr47kNNMAcT/g6PTSd31rfVW/Av6+73a7XKiQQxnyHOXclesc8r3hDaOH7AotM/6ourqQXdo0AAm3kJgF1EtWi2EoThMLHXCNraYpJNxQQjUr3tbNqHRsa48RLzgymqp8twyqB4T9FcZWUd9M9w9Jqo66RAw04rXxoLMQufS2m1XIlqttVAr/Ynl5GUSgCT+I8Em+47fUHji9pb0O+9ZgGFyoW7hsPMjNVfOHiWGMxh6tlQ/6uGLeCmFDco5StabvtxzX812vHGQ7e5UT2ZqbKah07kD0qD3lrODQWarmj2NBhW2M0qPneS+70urkAlq4ewIwyAjuQBPS2MBMJx5S2oOkkNzWFdhaYRw02FOzqVXb95ww04GhsJsye/2UZynObp2g48kStDyyhT5GL1ADk3u23utV4iDd3tbsdGWelbe+zfZnZhht18p3F8kQ2ERjCNaquGPIKWKQicOtooF34NvqqBE/dRlonjLl1ryZl/d+HdFMw35nXkIe+Piq52nvCo51L6/ti9RkKVX30ogC1tl3L4Dj3T6D7U5iqiQDm2mW4egDq57WzVyGdIp674PnJUxl2lG+DC3DG1EKKMRVo+e6781nyBfuTOO6mTpt21/serHvUQ8/rqe9IKe3RUOKFhtnARnj3ijAljvrH6o4/hmm5H+NVIE6D1neGA0MBv9wMOcbEnfNEaSrIZ7BR2rRxDTqY0GjIi6y3HfkgyjQnI7gEYqnreVYVGZbK52Hppd/bS5RsENgc+igedK3cHuFM0QPs7kGNiC+IXXZC5nJuV5W7rwA66QtlOodOBvw8sYGp36OE+BO5u2zuhUxgr+6RQFbQ6calzAgjVpVSszYuqiqunXGSN6oRh4h2Aj/cS6hcg5D0tuwr+qi7Ap0YDF1R6fIh4MyPlfvw4cUdwVOVWAWejCw9Old7JuzfKU39hGquctAwbqTPX0+dGMU2ljnvcGA90O6gk/aGfS+HdF/1BhA9PXCB5DxkluIiNQPJt2Kdigz/Ti9RSpaDoa2DZHOO87Aoc1HQ8RXZxgHUaVLZtwRz+HKbFIcSdaV9EPcJ4VgQqo7/VJ/TZDkMPpZ1DH2nUToaNMSBIeuxpcfu9a8kFKFP91/WIz3XM18Q/dPCUxqemRA1pFtzP5XDFpbUvxUuarpSwQ1yI4zbLAdPivcVttZpkMMTjc3IfvPLDH1ffTFlLdiHgSGFzKPEih8VukVPAXW8EZY0pKFR5lB+UeBgBybeiUk1iR0Nv3pCAOnb9VcEK86cmBQUIG8Nzr1WTEFo9xcbmH6aLztPKf3ToTBC2KdMk5Mqwd6bGrqRnsMxh5BovTNB2Tq9cgwP9sYUwM4zuUgBiry+52RSWbMYNQS/kyVe1KAmlJkCWSQ95zCCVlfF9SQXTlPer844d6gywHKZW3oYJ40ptiNZ7bXGAanWQXTGNrB/iGIWE8feNzT79GTnezydmBoT7kJKEB9M6troyOvZ14xEwoKvYS2L4NOfecbykB5mG2gel7aT8Ye835xhjg7yNu5F2opBrOqWjqJ86R15asv7ML+a7kjmI/9/s+OpxvZIq6E1XG7mn4BGE6+7TswtQ1OBeEszZjVLIqkGA1hJL3m4EiW1ecCl9TYIFIdpQZ9cdClgjc3WUl7DWTillrZQUt9DvY0Ad/p0MCkTpUfwMIONrSvpCjRdvg7AQoJbzaeWjatbN/IhrRriKf7B1As+wsDsyjW+A1He72+6Rx53+Hes05QXcd+ucFGhobHjPjBaCR4KJ9eUXMCjJ38mAzNaRmcqD4qRtN8AKs5SZokVMyaBIGXcVw3zXR1cWAWmPl9EL9O0EdCfY1VDn7n7oKs2NTHKqKIoSVCiB2YQC1rEqxoPtgPfm5cXNMN1YO4ItboEbargvrRem5PR3ck3rY4TgxeqGAjGCiSuoN/2wrThou0mozwSMG1w9PsOZtAtG7WO4FqOpVP17Gh/LYpIencMqhfgc0v9EX3cNK88+QZ9XLdA2nVBPPH3g+g1dMF6yyYdVrHBjAr+tuFTBpKV/jrYQplxfV3ouwvVugFtS5KMGDFb8pKtohZ/gqA1s9dy43LzmaIvmbD/uS8W8gJfjU4LmFQ9qgL7zoF1KwOwd7Yd76iPQQb6zd/WdxYPeI3+PgkfKx/2XMWPtYcTTTauSmv7kALAf3uT1NIKIww4U2xyhHe/PJvr5DSPw00q5txQjjNdYRif1Qku9erwm5CNbCYBO/CvgYrPtJuhQYbvvhgcHhQwdzyPtO8fxhsjBhijxJjv/1xkeKP1pJ/yrb8V4o/nz0KHGk9/VlA6ngHcniEmWXmNKczM870zEdDuH8Dt//fwO1Pgl1/WdgaAukXCFtPR5vzDti55gvA4i4ifnpM+pTo+AfafB4dnwXGz8XEN2qm3WqpeklGh6f2psPj5XP5WDdwuejFkzx+Efi9gevndPR82DbOIPhmS8H4MzxmQH3jzp8N1Z/PAXMAvyFxUX7bND8I6j8s5Gms/394QFmB', 'scripts/build_pairwise.py': 'eNqdVU2P2zYQvetXDHiSUUkFipy28CFIsoeg6C6QAD00BcEVRxYTmlRIyls3yH/vDCl5N7DjANHBH6OZNx/vcSiEeBVQJYSAcfIuYuuDxtAG5bTfm/9Qw+v7u/YQ23e372FSJjyayN49+cVOCFFVQ/B7kHKY0xxQSjD7yYcEyjmfVDKEWlWrLewmFSKWmEml0ZqHNeCe/i5oypqd26NLHR6UnTPKCXea0Gn5MXpnG3iYjdXywRoycXmxoeKUlkuFVVVpHGCvjKs3NxXQkwsIsD0V070Mu5mT3ec3tcbYBzNxzq2U2vdSbp5FdkprqZaQWrRtHJLgrJ9nE1Bv34cZGxjRTlvBQ9uhw1A6ePvu7s9f+eMPcRVRT/57iMTGzyBSja1TeyRYmoeabdoKrvtHZZwFcWnXMyFqCkjHCbfGpafQF79djZswEKOqjeloOaPqMwEiJk+qSjQC0eT4S0+Zzcv4CdKIkNUAH2e9Q0geVIwYIywJICcgIzyitfw9Rxxm69iHVA9RDZiO15v0c5rmM9ZLCPlFUtcSmb84NtabaoEklZLDmXLrU3vPFVxzbEdcbZoLdqJj8zSW1VUyaw2sHs//Mj0loFRrhmJfpiPzdG5OiIMPuWAwrhR+8w0HbPpb5EnLiQ7ulMQ/1Fp9RpS4JaA0mhMNhWfoKUYFQxaqz0a/sPY4IhEZMpvrZoKRuGKDOEdnGjAm2lZ8NHaYTE+rwc4uUQb/ybhd23vVj3Dwpmd4YxmY1wK9I631Mx0omtEF7CKPJteBfcq/V510cOcog2YQarBIB5ZqaBWVJvdzTEDL8BK6P2AIRvMkAjGbWIYNTMEcVH9sGJT2XfBkKFt6mKOyHbz2GdC4gaZE7VBrF8D3XqMlMK/nnmdj8lDXgXYf3Acn4JeLLH4Dtp6FNBK5vKaL+MopOOmI33f4r6HO112bxaz4xrilib/J796E4EM9iC/s/5VYZ1UfoQT+Dv3oPfkrcPj4dOGUVCXFcjSvSPP5FVFzTJM9ljYCrSXK/1fwNNEvFl2dwzdfy+KgQT2pMvICKZVSWrrHBrrp+DjRPbfdgpCSNSSlKMnLPVP9D4KPYR8=', 'scripts/plot_metrics.py': 'eNqtVt9P3DgQfs9fMeenRLebAtfqEFJOqnR3T1yLWtSX1SoyiZP14cSR7QBbxP9+M46zJMtCqbh9gNie+ebXN2Mzxi6UdvD170vgbQl/XnyGRjgjC5v+a3WroJJKWKi0AbcRuGq5AiM6bVzKGIuiyugG8rzqXW9EnoNs6AzBWu24k7q1UTTumbrjxopxTQYG/Y67jZJXo/IFLndaDXcduojHabelL+AWOuWiKCpFhb7wMveuxoRyBtaZBJZ/gJLWrUpZuPVZBPi7lW7jkb1ckupOtDEzbAGiLXQp2zpjvauWpywhCwgjeDOo0s8IDLCFFZlKlealjZVsReJTQ18g26AEsvI7KS5lFyfr4GrDZRsnA6RPhIFsl5T0o6n7RrTuwp/EpbAFalMGszwvdZHnyUQz5WWZ86ASs+XSVg5DQTO8Vy77pFuxgI1QXcaotrOasheByk4jEC+8ZcY7TFM5QV6tR9ynXImRGII7fqVE8rIRxa+Esq+zc06yRINig0UCwYsNeDc9bV62o3vX9W5ZSjOBZsOufUdssgEAtSyWI+D4f4Rk4+F4UEEBzyA6SIetHLGnImlzjTvIMYMu2OzS9FgIcYdkzPW1XyaRF0eSeBgs3IRl+pa8mJB6lEl2MtaJjoRWKLxitGJrz0JcEwkJY72TrmSNWb6j0JRLbX/lg45x28rvIot/X8D79EPyCM/vUhKJvZlFsOIMcjdX2toDthZYHHMtDCYWs+xLm7FL0mCPsDc4Nwaf01pgbXAtSz8gBtjk+RAoU+02Ro0eu8wCThYggnuNsNvSR/KYyAORoMBBT7/tPGGzLFh000mnxNBC3ssF3AWtz9iaDWbQ+HrgwTYcfBG2w6knlk5fixY+nZ/PYWsjy5irbsOzo/Tkw+xMiRq7IE6m1UudrDcuV3yLBNs7svxG4P84sPMdMGSKz2fatTVRvpPZ8enRoxaRoEABQQwITByacZhFNg2rX2FV+Ra/l20p7nB9/MB8xoc1VYm3tYgVjtGJYoJ5HXewR5NkPbKdtoMM/JLNpSYdwKUV8I2q+pcx2sTswugbWQqgio+TI5S9w/QPoyAMIn9dLQC91I3Em0KpEF0aijB2Heqc7feIsPtdcryA3xaw65Xj99QsJ5NmoXzQFApkoqx8x5E/mgjbdo+YT7vcX0kzmZ/r8oE/wq6O1v9H91bs3n88gNtr4ze28lva+bkYZ31NLUp3r3Vbatzl8kBMN4ca3qMXRW94sZ1FZ8QtN3irhLMfRYeO1HhwAGE4+ZG+j/B4HuFo+1Cp/N/kKcTJHGIw/gqAMcOT6UdT4LXT79wTINqPZoL2OG5hSMwuvNfgf9yVIdoP90UbIfmvsPBPKNOsx7kfeGTpyRXz7Dyfz/RK43OApsjpG6Y7jpR8fHL9xIDvjMTnUMW+ImAJfrRBj1PcwP2A/YDh4oO7wod8yxt6xmcZsDynF2uesyHm4fka/Qfld9YD', 'scripts/train_persona_lora.py': 'eNqdWF1vG7sRfdevINiHSq20iX2b4MLoFnDjpE3rNEbiPBkGQe9S0kb7dUmuHTfIf+8ZkrvLlWXdoH4wluTMcDgzPGcozvn1p0t2cfXxWsuiVpr9mV29fXfNLptP56xV2jS1XBn7WCqmu5rN7VYxaUyxqStVW6a+Zaq1RVMvEs75bLbWTcWEWHe200oIVlRtoy2Tdd1YSXJmNuvn9KaV2qh+/BVbef1W2m1Z3PXKVxgOWrbR2TbsI8vgRpJLK3vxspG5aLVaK63qTAndPJh9+arJVWkmGrbZqbr4r9Kz2ezDx4u3l+L9BUsZ/2e32RT15p3M1PXfX3yumvLyw+nql9cvP6ze18bqLrM49yxXayYwqjNplbDqm50PFpeMxmcM0ktWyW9+L3PGitou2OpvtHA2Y/hzC6LIDXYe1BMcA/7OyciSyTwXplVZIctgJ30nS6MWzkCxZqWq54OdBftrGm/phOhPK2Sodo7N4vGwaa78pr2lm7PRzO2SmV3R7vtxrTu44WPhUiuQYzNf+F3djMbB+sQn53rTUTqu3Mo8VybThaumVIi8yYRYRJoJnVwGlflwEL5aUfb5cpjB9rIrbcpp/kWoYVE2Wr7QsjLyUbh6jmrEJFR8kYmtKtuUX6PWW93cF7nK+7vARi32r88f/8PWRamC5vPewklXcnw5eNfX2FGlprNtZ1d5oSNN7mfN5Gj8+Oby2wp1sbFb2LGPrUpReqPFVyenv6uOQFStXflUH7Ry+ur1USulkrrGVVppXJHewhqXL/ZErV4dNXInbbZdGZToQR9OjipvtMwLfK9klnVVVzpEArqp9vCJfj1qTbVNtjXPHOSXo6pGqfzgjn85ngjKNMJX7/4Pd52uLNutPBy517+vneumRfE9c+SXycuQuwAmwVQMBQEdKnBNjwtWP46w5HCa7q1RdsDnCz+eCrVqbXuBSzj3pqnXxWbJrqXZXcO7qbTV5WDt6mMvO9Kek/Zkxt47ubdaNxpUR7MRbMrCKPapq21RKScy50QDsizZHjVq9VtXaEXfZgmoz8ouR/U7V2SduwMs2Z1aN1pBtzAMp9wlfOE9xrYzty0FDqg5iWKAepCqJ8Qk63KZFEbIe1mU8q5UfXSfc/oqwJnjeUtBIN/gROZiA/bO2ZoCwP4tNxuw/5svF+fsH1dfkoA0A1PAtymBzslJT7Fe9A/sTac1xQS9xh8NhR3blLh+2CCEyYC3kNAWdUeO0NYBcTL8LxXdVOYuP9B66kASlIQBUBNlkyXug0fk3zu41xN4N6nWvJc4+D2C4jBBllC6uXXz5ArkkUFnbQyr9w+Cz/M+FG64l+MgTR8Y0KifCry5GCxOXUhk26o6n38f1h3fBXNn7OY71w3I54zxDvcMt5LDgEWcMeWlftwup8rZtjGqnipTzaKCazu14HwP8rdPDGn1VWUWKPbTpgaNibEf/vThyiOY4bIndAlECWvzaVS8vGnLgqSDXuIqGOE3Vriluf9EFlLEe35CqegQy5fJKfuTa5GC5mKxQDMDOE5dcuhr4UuHEA87jNgy9hzaC5OEIDQej+OmHMhGIm68JxOgNJIKM6OcxZqyAhepK5VJb/hvVDdfKbS74et++Gr8121swOyEQ+oeFJM3518+n1+Kyw+j1F0hTcrrpp62MT0miIA/A2yOYfB9iEB34o8xjiPrykoww0l8fGO8VxxYWTVFHjVefY8gqEcIwYmnIkktDMCAYqODuRJILnVk7UHqqmtJsWiInqIVVWy2cFVl8nG6QteT+gRRN7pKT+KlBr1plXKZy+pBONyN9kIzBnP3BdDFF6MDK1+C7hzj+KCSukcj/TM6fRMj4iZGuCbGKx0RGK3UXRX89G2MV/Xf02D4vjEdsMuPR5l1e/Latf5R0teYcq+SA24jadmubdB8IKl7ioeFxO6Bdk+/E84JrSCgpQMWt8ePUd8FEa8pVMrmMeXuOFGOjLxXR5YdS9wRcDjyEtIK4O+ei5WyusgEWCEShS3amUqbx6dR2EkL8DKuAb72Y+L8sXgZQ7WoCrTQsTPu3Rm5G/rUAxKU+lfL6G1HDQwMT66123ACdNGTCWAonltED9PAz65G9HMB4u6q/tH5BDBcOzC2VSNW+DCNncFo3qV2gjYRgrkKDVCdOmS/4W4yxjkX+30h5CaWATbinWdoi6wETaUjS0e3cY3KcyCXEiQ/OR1qz6DTpce5P5/nndCQjfAHAfrRYr6HipM4JS75LhZz5Hgeab9gnEqLL/Y6La+CNsabyA/qHNjDVayZh8AtJ2dJwuoBPRC5VeFse1s5wvKR8k/nRfKgi74Bopkk76rWRL0Lv0N6/HlxcQ8VAg8p7JdpGK0Gr9GCQYAYfFoOi0gU9VDkHvYOyLvKiMWJoyAzqcGEJiOZCQv1Hj5DTdwRBWXM9UQH6YI/4S+SfcJgPKYwSEyoiscs9mRxQmRYnVAZD4/XcJB93Ocj8fQih6iIH6GaXu+n2IiPxDIUxwGq4U9651h6shAXzqSPgoYBOuL2UBEn08W4Ktwz+8x1a0kEEtw/ocPCof6O98/kWOZJd+eua8Rz4ZpSFexzyQ96PuaIYXqKZtX9GkjUyTu7Xv3a//CjYSO+eYcueWSHXuJ4QQpRy4p+qE3xbhKC3uVCcP/G8Y/02f8A8D0gwg==', 'scripts/judge_pairs_reliable.py': 'eNq1WVuP27YSfvevYNWHSoit7KYJUDjHAVqcFjgtcBAURV98DIGWaJtdiVJIar1bw//9zAxJXXzZzQapH7wWxblwLt/McKMo+l2YtuLrUrDf+HYLf6q6ECX7qy22Um3ZXtoda4Se5dwIpkVe3wv9yDa1ZhUv4U8lCrbmNt8Jk0ZRNJlsdF2xLNu0ttUiy5ismlpbxpWqLbeyVmYyCWt623BtRHj+y9Qq/K5N+GVl1e1oW1k4CQ23u1KuA/uP8Ohl81JuVSWUTcU9L1uSGbZpwYsMT6ELM2Vw8opr+bfI1qVURYaHRkLjGJlcy8aaFJdF1nCpTeCT0ZGzBrY1dsoyOoZbnEwmhdgw3jRCFTGqOSftpkzX+zkrZG4TNvvA/lsrMZ8w+OCeFDigytVdIXXsHsziD92KKRMP0tisvqPHhEicW5CuBjFxxCPYpvK6AJ8totZuZj9ECeOGGQtHrpwc/LjndK+lFTHaOy3aqjEx6IYcDPqMm1zKxS+8NCJhr1j0PxUlpww2ZWt2ceIP2xkyJivNWQkaL/GoqymrW9u0NhiBwmuObKaMWyvAfibrrUSmQUKncucRthj5LnZMEyY3nn9KVjJxwgTozZYroocAkEUmC6Q/DM+AB063wsaRLKIkoYCGJSZVL7PbD0JGJHuplNBg8ghI07LeCw1ygfRAjojW+GWliI7Ewn1X0hjMKFDEM1ui7NVINpnvyGa94kTr7PsItFdD1ll+2mvvTT1lGy5LdCswW3glnDs7868f4aULzSU5Riq7QkWd5nD8kac6U/dhhUeAiKrwDCM/jQgHBEMiEI50+BNNYqLVeN+5rktPhkqO35CD/Nspu8H4vR2acBm5bEY4ywJptHKWjU94IXwIOGdylYMWVkvRM6j4Q3wzZXndKgtevHXOdY9wxM/jj5yNZ110rB2TD+z2S1iCXlnvCqFJe2KNGl/hMWWQ27wt7eJmxI58CaQ+8RCLMsUr4ZMyNRgIgBuZ358izkTnLBwKZVY82CEU+S0YhQUE8uJNcgHbHHBqCNSnSd1GsGWrVRDuQcthNtSgLpJpRWNM+cKU/qi3LWbTR3oTF8LVBKgoiywr6jzLkgFlyosCGRJJHM1mlJMABlp8aqUWxQDBr1A4E76IhNL8RRRUp2YGUATI7GMjFmDH3tvvniQ2pRDNzEB+q8IE+k1Z8yGH9OZplfnDLKTOJQ2+f5JaqPvZRpaofCCIXtuqed3wN7M1hMoOvHxnUtgXjfzv+Q0d72Oh4lKFKMAXEAOjXQEG4Sl1pR+tx/4F2QjJSMuYYv5Mo3WyV+btBW/e9eCmuYRS9Sem289a1zqOfkLejHhzVbDAr2qNZWvBmtpIK+/Fe0Zcu3VuWSk4/H7HgmPgZISx2MgUtQVbhN4FXFVkbsntGSzEpDL8yNDAULuh29OyEIOY8lzuqFvMenNjt3FHj5ORuVw/Cc0fopXbkJZlZU7N8DsAGnR6zhCb6FcEWXboWXyjj6xV/B6qGbar71m+q2UuoNk4GFBIFPGAeXL0niesBm8O3i17nisfZtjZnTQYtIneJN0m30hgjaRyOyzg+NxV8FWwQAnNmYGKFKih0/hmQavdylPh8BGZ/ufffQC0Sn5qhT+bwwpQCBsnp7BviwavX9hYhmbhcyHe7zeE8WV00lh8LpdAMOJC2rmO6UtavwL66xd0fUHa1236cAZwR1hSgJxFShAzDCefK3iA1aDOXciJIwQ/BhNuTY7M8HsBXQ8tecmw6n+l3qwl4ESGtUIYjB0cQxxGtgobsQXNWCl+vY2TdCcelvMfVi6lESk9ZUwoOG70IS99az8IagUOrXNejuQOjTzSRxo6fT8bhQ8clj+6ynwKqTMWI3CARYC0VjIHt8xGbJMRK5DpuH1gN+eNJrEiATHt6mlPDHcqsz/T8yBBxlt1FH5WHKUNuJF64Tnyo8iqBDyQVPyKk2MvksZBD3H5jkMiKbGHgMGCeHCOPc4O5B8AxvGhNd9jhwnaElambqaNRxOu83aS9BJ9RR0OvjFwgpka+28Euj5kMCxj2jgaMmzblML9hK/VqpsHM2pu5ziFXBgxqLGPXTZi08MBMhKXTyghxT2j2WRgYYcX0wEohA86ad7zD9k+Z0FQ5CTBivsxPWfh5gLXjs0HxQ+Iu0ljPj7kmMtxHKiY1dj8hHkmlHRueZlhzRVUtuLo7c0tgs/bmzfuz/f4B2pla3c1zooFPoP91rKAthgfPrW15YB5uRCFey0VjZxgKcnuxKMHC6sfxx5wUYHt9QO6jWuCKqGgOdPcCogBtRU4Ag0xaHraOMHK7YmH6CW2XY5mSbznTsKrU/rViJQSU2DFHOJZ+LiwuPQGz+O9QXMrqX47PW/nXp1pe2abixJHWAmDeHQYmO/o2/CDV+EYJRe5DbKny5DLO9cQmXdnb9DJcMCf6Q9ehUGfBmuXVR+ZEvTF1jyG3UmaUeXOMiw5sHCMLtJj06ceYxecaNSeYVcy3d1h2DCK5uSyVs80iTyHNtBQ64IIc+hlgk1dBwwan9YA7yWoOOfVZgAZw4boEnB48MDM/gzIn16mb3Y4js+hicBdmI6kPqz0Z7lCOkSWk7g9pzieB8637MfTm1xmdnVbFlSJC2lyrgtW253Qe3QBoRxdGNM9RXrGka4n8HLEo0SwQA8SPh9ur7g73AktTjuj50P2LMe/Uq4/m/NXc5/6vtV58qOFZofeTsdngeDlgPA0MHwRQHwtoPjnAeOfAY7OaDBpXJf9Auw4xZBwszkdoAJe2eJ1x9MMLkDGC+HhCkyE8eMXqaTZdSgxiufRIPL6ECZcGMTdVXZ/Lx3mkI1UvCwH6XThkjtcaoe+bdhTjW+Ww8Dd3Xi26k7VezW8S74YCYdA8Z2n+G51HPzboVUw3NYlzFXvIfhMI3LLDkO/Hv1FDXTZeOcyAR1CBrDFAmbbDO+WsixyeriLpsn/AfPcg7k='}
for relative, encoded in PATCHED_FILES.items():
    target = REPO / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(zlib.decompress(base64.b64decode(encoded)))

sys.path.insert(0, str(REPO))
commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
(OUT / "source_commit.txt").write_text(commit + "\n", encoding="utf-8")
print("Fork commit:", commit)
print("Patched files:", list(PATCHED_FILES))
print("Outputs:", OUT)


In [ ]:
import importlib.metadata, json, subprocess, sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
assert torch.cuda.device_count() >= 2, "PA2's DPO configuration needs two visible GPUs"
print(json.dumps({
    "torch": torch.__version__,
    "transformers": importlib.metadata.version("transformers"),
    "datasets": importlib.metadata.version("datasets"),
    "protobuf": importlib.metadata.version("protobuf"),
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}, indent=2))

subprocess.run([sys.executable, "-m", "scripts.check_environment"], cwd=REPO, check=True)
smoke = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py"],
                       cwd=REPO, text=True, capture_output=True, check=True)
print(smoke.stdout)

ENV_FILE = Path("/tmp/pa2-benchmarks.env")
PREFERRED_JUDGES = [
    "google/gemini-3.6-flash",
    "google/gemini-3-flash-preview",
    "google/gemini-3.1-flash-lite-preview",
    "anthropic/claude-haiku-4-5@20251001",
]

def refresh_judge_credentials():
    result = subprocess.run(
        ["kaggle", "benchmarks", "auth", "-y", "--env-file", str(ENV_FILE)],
        cwd="/tmp", text=True, capture_output=True,
    )
    if result.returncode:
        raise RuntimeError("Kaggle Benchmarks authentication failed: " + result.stderr[-800:])
    if not ENV_FILE.is_file():
        raise RuntimeError("Kaggle Benchmarks did not create its credential file")

refresh_judge_credentials()
probe = r'''from dotenv import load_dotenv
load_dotenv("/tmp/pa2-benchmarks.env", override=True)
import kaggle_benchmarks as kbench
import json
print("PA2_MODELS=" + json.dumps(sorted(kbench.llms)))'''
result = subprocess.run([sys.executable, "-c", probe], cwd=REPO, text=True, capture_output=True, check=True)
available = json.loads(next(line.split("=", 1)[1] for line in result.stdout.splitlines() if line.startswith("PA2_MODELS=")))
JUDGE_MODEL = next((model for model in PREFERRED_JUDGES if model in available), None)
if JUDGE_MODEL is None and available:
    JUDGE_MODEL = available[0]
if not JUDGE_MODEL:
    raise RuntimeError(f"No supported Kaggle judge is available. Models: {available}")
print("Selected judge:", JUDGE_MODEL)

ping = r'''from dotenv import load_dotenv
load_dotenv("/tmp/pa2-benchmarks.env", override=True)
import kaggle_benchmarks as kbench
import sys, uuid
with kbench.chats.new("pa2-preflight-" + uuid.uuid4().hex[:8]):
    response = kbench.llms[sys.argv[1]].prompt("Reply with exactly OK.")
print("PA2_JUDGE_PING=" + str(response)[:80])'''
result = subprocess.run([sys.executable, "-c", ping, JUDGE_MODEL], cwd=REPO, text=True, capture_output=True)
if result.returncode or "PA2_JUDGE_PING=" not in result.stdout:
    raise RuntimeError("Judge request failed before training: " + result.stderr[-1200:])
print(next(line for line in result.stdout.splitlines() if line.startswith("PA2_JUDGE_PING=")))
(OUT / "preflight.json").write_text(json.dumps({
    "public_adapter_tests_passed": True,
    "public_test_output": smoke.stdout.strip(),
    "two_gpus_visible": torch.cuda.device_count() >= 2,
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "judge_model": JUDGE_MODEL,
    "judge_ping_succeeded": True,
}, indent=2), encoding="utf-8")


In [ ]:
import json, subprocess, sys, time
from pathlib import Path

STATUS_PATH = OUT / "run_status.json"
STATUS = json.loads(STATUS_PATH.read_text(encoding="utf-8")) if STATUS_PATH.exists() else {}

def record(name, state, detail=""):
    STATUS[name] = {"state": state, "detail": str(detail), "time": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())}
    STATUS_PATH.write_text(json.dumps(STATUS, indent=2), encoding="utf-8")
    print(f"[{name}] {state}: {detail}")

def run_cmd(name, args):
    log_path = OUT / "logs" / f"{name}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running", name, "→", log_path, flush=True)
    started = time.monotonic()
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(args, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
        while True:
            try:
                rc = proc.wait(timeout=60)
                break
            except subprocess.TimeoutExpired:
                elapsed = (time.monotonic() - started) / 60
                print(f"  {name}: {elapsed:.0f} min elapsed", flush=True)
    tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-24:]
    print("\n".join(tail), flush=True)
    if rc:
        raise RuntimeError(f"{name} exited {rc}; details in {log_path}")
    return log_path

def module(name, module_name, *args):
    return run_cmd(name, [sys.executable, "-m", module_name, *map(str, args)])

def safe_stage(name, work, ready):
    try:
        if ready():
            record(name, "complete", "Verified existing files")
            return True
        work()
        if not ready():
            raise RuntimeError("Command returned without complete output artifacts")
        record(name, "complete", "Artifacts verified")
        return True
    except Exception as exc:
        record(name, "failed", repr(exc))
        return False

def count_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return 0
    with path.open(encoding="utf-8") as f:
        return sum(bool(line.strip()) for line in f)

def full_model(path):
    path = Path(path)
    return path.is_dir() and (path / "config.json").is_file() and (any(path.glob("*.safetensors")) or any(path.glob("*.bin")))

def lora_model(path):
    path = Path(path)
    return (path / "adapter_config.json").is_file() and (path / "adapter_model.safetensors").is_file()

def dedupe_identical_weights(best, final):
    # One-epoch best/final weights are identical. Hard links keep both valid
    # checkpoint paths while using one disk copy when the filesystem permits it.
    import hashlib, os
    def digest(path):
        h = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                h.update(block)
        return h.digest()
    saved = 0
    for source in Path(best).glob("*.safetensors"):
        target = Path(final) / source.name
        if not target.is_file() or source.stat().st_size != target.stat().st_size:
            continue
        if source.stat().st_ino == target.stat().st_ino or digest(source) != digest(target):
            continue
        link = target.with_name(target.name + ".dedup-link")
        try:
            os.link(source, link)
            link.replace(target)
            saved += source.stat().st_size
        except OSError as exc:
            print("Checkpoint deduplication skipped:", exc)
            if link.exists():
                link.unlink()
    if saved:
        print(f"Reused identical best/final model weights: {saved / 2**30:.2f} GiB saved")

BASE = "Qwen/Qwen2.5-0.5B"
PERSONA_BASE = "HuggingFaceTB/SmolLM2-360M-Instruct"
HF_UF = "HuggingFaceH4/ultrafeedback_binarized"
HF_HH = "Anthropic/hh-rlhf"

RUN_CONFIG = {
    "fork": REPO_URL, "commit": commit, "judge_model": JUDGE_MODEL,
    "base_model": BASE, "persona_base_model": PERSONA_BASE,
    "sft_data": {"source": HF_UF, "train_split": "train_sft", "validation_split": "test_sft", "train_examples": 30000, "validation_examples": 1000},
    "hh_data": {"source": HF_HH, "train_split": "train", "validation_split": "test", "train_pairs": 15000, "validation_pairs": 1000},
    "uf_data": {"source": HF_UF, "train_split": "train_prefs", "validation_split": "test_prefs", "train_pairs": 15000, "validation_pairs": 1000},
    "eval": {"alpaca_source": "tatsu-lab/alpaca_eval/alpaca_eval.json", "alpaca_seed": 42, "alpaca_count": 300, "hh_count": 300, "persona_count": 8},
    "decoding": {"do_sample": False, "temperature": None, "top_p": None, "max_new_tokens": 256, "repetition_penalty": 1.1, "no_repeat_ngram_size": 3, "length_penalty": 1.0},
}
(OUT / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")


## Fixed prompts and baseline

The first run creates the seed-42 AlpacaEval subset, copies the supplied HH and persona sets into the output, and generates Qwen base responses. The same fixed files are used by every comparison. A three-prompt sanity pass checks decoding before full generation.


In [ ]:
import shutil
from huggingface_hub import hf_hub_download
from alignment.evaluation import read_records

ALPACA = OUT / "eval_sets" / "alpaca_eval_300_seed42.json"
HH_EVAL = OUT / "eval_sets" / "hh_rlhf_eval_300_seed42.json"
PERSONA_EVAL = OUT / "eval_sets" / "persona_style_prompts.json"

def prepare_prompts():
    ALPACA.parent.mkdir(parents=True, exist_ok=True)
    source = hf_hub_download(repo_id="tatsu-lab/alpaca_eval", filename="alpaca_eval.json", repo_type="dataset")
    module("alpaca_subset", "scripts.make_eval_subset", "--source", source, "--count", 300, "--seed", 42, "--output", ALPACA)
    shutil.copy2(REPO / "eval_sets/hh_rlhf_eval_300_seed42.json", HH_EVAL)
    shutil.copy2(REPO / "eval_sets/persona_style_prompts.json", PERSONA_EVAL)
    assert len(read_records(ALPACA)) == 300
    assert len(read_records(HH_EVAL)) == 300
    assert len(read_records(PERSONA_EVAL)) == 8

safe_stage("fixed_prompts", prepare_prompts,
           lambda: all(p.is_file() for p in (ALPACA, HH_EVAL, PERSONA_EVAL))
           and len(read_records(ALPACA)) == 300
           and len(read_records(HH_EVAL)) == 300
           and len(read_records(PERSONA_EVAL)) == 8)

def generate(name, model, prompts, count, *, dataset, chat=False):
    output = OUT / "generations" / f"{name}.jsonl"
    def work():
        if not Path(prompts).is_file():
            raise FileNotFoundError(prompts)
        rows = read_records(prompts)
        sanity = OUT / "eval_sets" / f"{name}_sanity3.json"
        sanity.write_text(json.dumps(rows[:3], indent=2, ensure_ascii=False), encoding="utf-8")
        base_args = ["--model", model, "--generator", name, "--dataset", dataset,
                     "--device", "cuda:0", "--max-new-tokens", "256",
                     "--repetition-penalty", "1.1", "--no-repeat-ngram-size", "3", "--resume"]
        if chat:
            base_args.append("--use-chat-template")
        sample_output = OUT / "sanity" / f"{name}.jsonl"
        module(f"sanity_{name}", "scripts.generate", "--prompts", sanity, "--output", sample_output, *base_args)
        sample_rows = read_records(sample_output)
        if len(sample_rows) != 3 or any(not row.get("output", "").strip() for row in sample_rows):
            raise RuntimeError(f"{name} sanity generation is empty or incomplete")
        print(f"{name} sanity sample:", sample_rows[0]["output"][:250])
        module(f"generate_{name}", "scripts.generate", "--prompts", prompts, "--output", output, *base_args)
    safe_stage(f"generate_{name}", work, lambda: count_jsonl(output) == count)
    return output

BASE_ALPACA = generate("base_alpaca", BASE, ALPACA, 300, dataset="alpaca_eval")


## Train the four required tracks

SFT uses 30,000 UltraFeedback SFT examples and 1,000 held-out examples. Both from-scratch DPO policies start from that **same best SFT checkpoint** and use separate HH and UltraFeedback preference pairs. Persona uses the provided 220-row data with TRL and rank-8 LoRA. The script logs training loss every 100 optimizer updates and writes best/final checkpoints. Identical one-epoch best/final weight files are hard linked when possible to stay within Kaggle's output disk limit. Each stage records completion or failure in `outputs/run_status.json`.


In [ ]:
SFT_BEST = OUT / "sft" / "best"
SFT_FINAL = OUT / "sft" / "final"

safe_stage("train_sft", lambda: module(
    "train_sft", "scripts.train_sft",
    "--train-source", HF_UF, "--train-split", "train_sft",
    "--validation-source", HF_UF, "--validation-split", "test_sft",
    "--model", BASE, "--output-dir", OUT / "sft",
    "--train-examples", 30000, "--validation-examples", 1000,
    "--max-length", 512, "--batch-size", 2,
    "--gradient-accumulation-steps", 16, "--learning-rate", "2e-5",
    "--weight-decay", 0.1, "--warmup-ratio", 0.03,
    "--epochs", 1, "--seed", 42,
), lambda: full_model(SFT_BEST) and full_model(SFT_FINAL)
   and (OUT / "sft/metrics.jsonl").is_file())
if full_model(SFT_BEST) and full_model(SFT_FINAL):
    dedupe_identical_weights(SFT_BEST, SFT_FINAL)


In [ ]:
HH_BEST = OUT / "dpo_hh" / "best"
HH_FINAL = OUT / "dpo_hh" / "final"

def train_hh():
    if not full_model(SFT_BEST):
        raise FileNotFoundError("Best SFT checkpoint is missing")
    module("train_dpo_hh", "scripts.train_dpo",
           "--track", "hh", "--train-source", HF_HH, "--train-split", "train",
           "--validation-source", HF_HH, "--validation-split", "test",
           "--sft-checkpoint", SFT_BEST, "--output-dir", OUT / "dpo_hh",
           "--train-pairs", 15000, "--validation-pairs", 1000,
           "--max-length", 512, "--max-prompt-length", 256,
           "--batch-size", 1, "--gradient-accumulation-steps", 32,
           "--learning-rate", "1e-6", "--beta", 0.1, "--epochs", 1,
           "--policy-device", "cuda:0", "--reference-device", "cuda:1",
           "--optimizer", "adamw", "--seed", 42)

safe_stage("train_dpo_hh", train_hh,
           lambda: full_model(HH_BEST) and full_model(HH_FINAL)
           and (OUT / "dpo_hh/metrics.jsonl").is_file())
if full_model(HH_BEST) and full_model(HH_FINAL):
    dedupe_identical_weights(HH_BEST, HH_FINAL)


In [ ]:
UF_BEST = OUT / "dpo_ultrafeedback" / "best"
UF_FINAL = OUT / "dpo_ultrafeedback" / "final"

def train_uf():
    if not full_model(SFT_BEST):
        raise FileNotFoundError("Best SFT checkpoint is missing")
    module("train_dpo_ultrafeedback", "scripts.train_dpo",
           "--track", "ultrafeedback", "--train-source", HF_UF, "--train-split", "train_prefs",
           "--validation-source", HF_UF, "--validation-split", "test_prefs",
           "--sft-checkpoint", SFT_BEST, "--output-dir", OUT / "dpo_ultrafeedback",
           "--train-pairs", 15000, "--validation-pairs", 1000,
           "--max-length", 512, "--max-prompt-length", 256,
           "--batch-size", 1, "--gradient-accumulation-steps", 32,
           "--learning-rate", "1e-6", "--beta", 0.1, "--epochs", 1,
           "--policy-device", "cuda:0", "--reference-device", "cuda:1",
           "--optimizer", "adamw", "--seed", 42)

safe_stage("train_dpo_ultrafeedback", train_uf,
           lambda: full_model(UF_BEST) and full_model(UF_FINAL)
           and (OUT / "dpo_ultrafeedback/metrics.jsonl").is_file())
if full_model(UF_BEST) and full_model(UF_FINAL):
    dedupe_identical_weights(UF_BEST, UF_FINAL)


In [ ]:
PERSONA_BEST = OUT / "persona_lora" / "best"

safe_stage("train_persona_lora", lambda: module(
    "train_persona_lora", "scripts.train_persona_lora",
    "--data", REPO / "data/persona_lora/ramsay_style_preferences.json",
    "--model", PERSONA_BASE, "--output-dir", OUT / "persona_lora",
    "--max-length", 512, "--max-prompt-tokens", 256,
    "--learning-rate", "5e-5", "--batch-size", 1,
    "--gradient-accumulation-steps", 8, "--epochs", 3,
    "--seed", 42, "--lora-rank", 8, "--lora-alpha", 16,
    "--lora-dropout", 0.05,
), lambda: lora_model(PERSONA_BEST)
   and (OUT / "persona_lora/lora_config.json").is_file()
   and (OUT / "persona_lora/trainer_state.json").is_file())


## Plots and matched generations

Every comparison uses the same prompt file and greedy decoding settings for both checkpoints. HH uses the provided 300 safety prompts, UltraFeedback uses the seed-42 300 AlpacaEval subset, and persona uses the provided eight style prompts against its own SmolLM2 base.


In [ ]:
def make_plots():
    for required in (OUT / "sft/metrics.jsonl", OUT / "dpo_hh/metrics.jsonl", OUT / "dpo_ultrafeedback/metrics.jsonl"):
        if not required.is_file():
            raise FileNotFoundError(required)
    module("plot_metrics", "scripts.plot_metrics",
           "--sft", OUT / "sft/metrics.jsonl",
           "--dpo", OUT / "dpo_hh/metrics.jsonl", "--labels", "HH-RLHF",
           "--dpo", OUT / "dpo_ultrafeedback/metrics.jsonl", "--labels", "UltraFeedback",
           "--output-dir", OUT / "plots")
    state = OUT / "persona_lora/trainer_state.json"
    if state.is_file():
        import matplotlib.pyplot as plt
        history = json.loads(state.read_text(encoding="utf-8")).get("log_history", [])
        train = [(r["step"], r["loss"]) for r in history if "loss" in r and "step" in r]
        valid = [(r["step"], r["eval_loss"]) for r in history if "eval_loss" in r and "step" in r]
        fig, ax = plt.subplots(figsize=(7, 4.3))
        if train: ax.plot(*zip(*train), label="Train")
        if valid: ax.plot(*zip(*valid), marker="o", label="Validation")
        ax.set(title="Persona LoRA-DPO loss", xlabel="Optimizer step", ylabel="Loss")
        ax.grid(alpha=0.25); ax.legend(); fig.tight_layout()
        fig.savefig(OUT / "plots/persona_loss.png", dpi=180); plt.close(fig)

safe_stage("plots", make_plots,
           lambda: (OUT / "plots/sft_loss.png").is_file()
           and (OUT / "plots/dpo_metrics.png").is_file()
           and (OUT / "plots/persona_loss.png").is_file())


In [ ]:
SFT_HH = generate("sft_hh", str(SFT_BEST), HH_EVAL, 300, dataset="hh_rlhf_style_eval")
DPO_HH = generate("dpo_hh_hh", str(HH_BEST), HH_EVAL, 300, dataset="hh_rlhf_style_eval")
SFT_ALPACA = generate("sft_alpaca", str(SFT_BEST), ALPACA, 300, dataset="alpaca_eval")
DPO_UF = generate("dpo_ultrafeedback_alpaca", str(UF_BEST), ALPACA, 300, dataset="alpaca_eval")
PERSONA_BASE_GEN = generate("persona_base_style", PERSONA_BASE, PERSONA_EVAL, 8,
                            dataset="persona_style_eval", chat=True)
PERSONA_LORA_GEN = generate("persona_lora_style", str(PERSONA_BEST), PERSONA_EVAL, 8,
                            dataset="persona_style_eval", chat=True)


In [ ]:
PAIR_SPECS = [
    ("hh", SFT_HH, DPO_HH, "sft", "dpo_hh", 300),
    ("ultrafeedback", SFT_ALPACA, DPO_UF, "sft", "dpo_ultrafeedback", 300),
    ("persona", PERSONA_BASE_GEN, PERSONA_LORA_GEN, "smollm2_base", "persona_lora", 8),
]
PAIR_PATHS = {}
for name, left, right, left_name, right_name, expected in PAIR_SPECS:
    path = OUT / "pairs" / f"{name}_blind.jsonl"
    PAIR_PATHS[name] = path
    def build(left=left, right=right, left_name=left_name, right_name=right_name, path=path, expected=expected, name=name):
        if count_jsonl(left) != expected or count_jsonl(right) != expected:
            raise RuntimeError(f"{name} generations are incomplete")
        args = ["--sft", left, "--dpo", right,
                "--sft-name", left_name, "--dpo-name", right_name,
                "--seed", 42, "--output", path]
        if name == "persona":
            args.append("--persona-style")
        module(f"pairs_{name}", "scripts.build_pairwise", *args)
    safe_stage(f"pairs_{name}", build, lambda path=path, expected=expected: count_jsonl(path) == expected)


## Kaggle judge — resumable and blinded

The judge sees only anonymous Response A/B text. Requests use batches of five with at least five seconds between calls and three retries. If a whole batch cannot be parsed, the notebook retries its cases individually. Each valid result is appended immediately; failure and attempt logs are separate for each evaluation. The selected judge model is kept the same across all three sets.


In [ ]:
JUDGMENT_PATHS = {}
for name, _left, _right, _left_name, _right_name, expected in PAIR_SPECS:
    pairs = PAIR_PATHS[name]
    output = OUT / "judgments" / f"{name}.jsonl"
    summary_path = output.with_name(output.stem + "_summary.json")
    JUDGMENT_PATHS[name] = output
    def judge_work(name=name, pairs=pairs, output=output, expected=expected):
        if count_jsonl(pairs) != expected:
            raise RuntimeError(f"{name} pair file is incomplete")
        refresh_judge_credentials()
        module(f"judge_{name}", "scripts.judge_pairs_reliable",
               "--pairs", pairs, "--output", output,
               "--model", JUDGE_MODEL, "--batch-size", 5,
               "--sleep-seconds", 5, "--max-retries", 3,
               "--env-file", ENV_FILE)
    def judgment_ready(summary_path=summary_path, expected=expected):
        if not summary_path.is_file():
            return False
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        return summary.get("judged_count") == expected and summary.get("unknown") == 0
    safe_stage(f"judge_{name}", judge_work, judgment_ready)

all_summaries = {}
for name in JUDGMENT_PATHS:
    path = (OUT / "judgments" / f"{name}_summary.json")
    if path.is_file():
        all_summaries[name] = json.loads(path.read_text(encoding="utf-8"))
(OUT / "evaluation_summary.json").write_text(json.dumps(all_summaries, indent=2), encoding="utf-8")
print(json.dumps({name: {k: summary.get(k) for k in ("sample_count", "judged_count", "wins", "ties", "losses", "unknown", "failure_rate")} for name, summary in all_summaries.items()}, indent=2))


## Report PDF and compact evidence bundle

The report is built from the run's actual metrics and judgments. It includes the repository and data, measured training settings, training figures, three separate evaluation rows, retry/failure counts, representative judged cases, and limitations. Review the PDF before submission. The evidence ZIP excludes large model weights; the full checkpoints stay in the saved notebook output.


In [ ]:
try:
    from html import escape
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

    REPORT = OUT / "report" / "PA2_final_report.pdf"
    REPORT.parent.mkdir(parents=True, exist_ok=True)
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="SmallPA", parent=styles["BodyText"], fontSize=8, leading=10, spaceAfter=5))
    styles.add(ParagraphStyle(name="TinyPA", parent=styles["BodyText"], fontSize=7, leading=8.5, spaceAfter=3))
    styles.add(ParagraphStyle(name="TitlePA", parent=styles["Title"], fontSize=17, leading=20, alignment=TA_CENTER))

    def P(text, tiny=False):
        return Paragraph(escape(str(text)).replace("\n", "<br/>"), styles["TinyPA" if tiny else "SmallPA"])

    def section(title):
        return Paragraph(title, styles["Heading2"])

    def read_metric(path):
        if not path.is_file(): return None
        rows = read_records(path)
        return rows[-1] if rows else None

    def fmt(value, digits=3):
        return "—" if value is None else f"{value:.{digits}f}"

    summaries = json.loads((OUT / "evaluation_summary.json").read_text(encoding="utf-8"))
    sft_metric = read_metric(OUT / "sft/metrics.jsonl")
    hh_metric = read_metric(OUT / "dpo_hh/metrics.jsonl")
    uf_metric = read_metric(OUT / "dpo_ultrafeedback/metrics.jsonl")
    persona_state_path = OUT / "persona_lora/trainer_state.json"
    persona_state = json.loads(persona_state_path.read_text(encoding="utf-8")) if persona_state_path.is_file() else {}
    persona_history = persona_state.get("log_history", [])
    persona_eval_losses = [row["eval_loss"] for row in persona_history if "eval_loss" in row]
    complete_eval = all(summaries.get(name, {}).get("unknown") == 0 and summaries.get(name, {}).get("judged_count") == n
                        for name, n in (("hh", 300), ("ultrafeedback", 300), ("persona", 8)))
    title_suffix = "" if complete_eval else " — INCOMPLETE EVALUATION"

    story = [Paragraph("PA2: Small-Model Alignment" + title_suffix, styles["TitlePA"]), Spacer(1, 9)]
    story += [section("1. Repository, models, and data"),
              P("Repository: " + REPO_URL + " ; source commit: " + commit),
              P("Base model: Qwen/Qwen2.5-0.5B. SFT: HuggingFaceH4/ultrafeedback_binarized train_sft/test_sft, 30,000/1,000 single-turn rows. HH DPO: Anthropic/hh-rlhf train/test, 15,000/1,000 pairs. UltraFeedback DPO: train_prefs/test_prefs, 15,000/1,000 pairs. Persona: supplied 220-row Ramsay-style preferences with a seeded 80/20 split and SmolLM2-360M-Instruct."),
              P("Fixed evaluation: supplied 300 HH helpful/harmless prompts; 300 AlpacaEval instructions sampled with seed 42; supplied 8 persona prompts covering cooking, coding, help, and safety. Multi-turn rows are filtered by the single-turn data loader. SFT, HH DPO, and UltraFeedback DPO use the same plain instruction template and seed 42."),
              section("2. Actual experimental configuration")]

    config_rows = [[P(x, True) for x in ("Track", "Strategy", "Optimizer / LR", "Batch × accum.", "Length / epochs", "β / reference")],
     [P(x, True) for x in ("SFT", "full FP32 + fp16 autocast", "AdamW 2e-5; cosine; 3% warmup; wd .1; clip 1", "2 × 16 = 32", "512 / 1", "—")],
     [P(x, True) for x in ("HH DPO", "full FP32 + fp16 autocast", "AdamW 1e-6 fixed; wd 0; clip 1", "1 × 32 = 32", "512 (prompt 256) / 1", "0.1 / SFT best")],
     [P(x, True) for x in ("UF DPO", "full FP32 + fp16 autocast", "AdamW 1e-6 fixed; wd 0; clip 1", "1 × 32 = 32", "512 (prompt 256) / 1", "0.1 / SFT best")],
     [P(x, True) for x in ("Persona", "LoRA r8 α16 dropout .05; q/k/v/o", "AdamW via TRL; 5e-5; epoch evaluation", "1 × 8 = 8", "512 (prompt 256) / 3", "0.1 / SmolLM2 base")]]
    table = Table(config_rows, colWidths=[0.72*inch, 1.15*inch, 1.42*inch, 0.75*inch, 0.86*inch, 1.05*inch], repeatRows=1)
    table.setStyle(TableStyle([("GRID", (0,0), (-1,-1), .3, colors.grey), ("BACKGROUND", (0,0), (-1,0), colors.lightgrey), ("VALIGN", (0,0), (-1,-1), "TOP"), ("LEFTPADDING", (0,0), (-1,-1), 3), ("RIGHTPADDING", (0,0), (-1,-1), 3)]))
    story += [table, Spacer(1, 6),
              P("All runs use seed 42. SFT and both full DPO policies keep FP32 trainable weights with FP16 autocast and gradient scaling. SFT uses response-token mean NLL, cosine decay and 3% warmup. Full DPO uses AdamW, fixed learning rate, no warmup, weight decay 0, and gradient clipping at 1.0. The policy is on cuda:0 and its frozen FP16 SFT-best reference is on cuda:1. Persona uses TRL DPOTrainer with AdamW, linear decay, zero warmup, weight decay 0, clipping at 1.0, FP16, and a same-base reference. Best SFT and persona checkpoints minimize validation loss; full DPO checkpoints maximize validation reward accuracy."),
              P("Prompt template for Qwen: ### Instruction: [instruction], blank line, ### Response:. The same format is used in SFT, full DPO, and Qwen generation. Decoding for every matched pair: greedy (temperature/top-p unused), 256 new tokens, repetition penalty 1.1, no-repeat 3-gram, length penalty 1.0, EOS/pad from the tokenizer. Persona uses its chat template on both sides."),
              P("Parameter choices: the guide allows deterministic anti-repetition controls; repetition penalty 1.1 and no-repeat 3-gram were selected after repetitive pilot outputs. A controlled decoding ablation was not run, so their effect on measured win rate is unknown. The persona guide leaves LoRA values open; rank 8, alpha 16, dropout .05, and q/k/v/o attention targets keep the trainable adapter small for T4 memory. Batch 1 with accumulation 8 gives effective batch 8 for the 220-row style set, and three epochs allow repeated exposure. These choices were not compared against other persona configurations. The full-model settings follow the guide. Validation ran at epoch boundaries; training loss was logged every 100 updates.")]

    story += [PageBreak(), section("3. Training behavior")]
    if sft_metric:
        story.append(P(f"SFT: {sft_metric['step']} optimizer updates; final train loss {fmt(sft_metric.get('train_loss'))}; validation loss {fmt(sft_metric.get('validation_loss'))}."))
    if hh_metric and uf_metric:
        story.append(P(f"HH DPO: {hh_metric['step']} updates, validation reward accuracy {fmt(hh_metric.get('reward_accuracy'))}, margin {fmt(hh_metric.get('reward_margin'))}. UltraFeedback DPO: {uf_metric['step']} updates, validation reward accuracy {fmt(uf_metric.get('reward_accuracy'))}, margin {fmt(uf_metric.get('reward_margin'))}."))
    if persona_eval_losses:
        story.append(P(f"Persona LoRA-DPO: {persona_state.get('global_step', '—')} updates; best logged evaluation loss {fmt(min(persona_eval_losses))}."))
    for filename, width, height in (("sft_loss.png", 5.7, 3.2), ("dpo_metrics.png", 6.5, 2.2), ("persona_loss.png", 5.7, 2.2)):
        path = OUT / "plots" / filename
        if path.is_file():
            story.append(Image(str(path), width=width*inch, height=height*inch))
            story.append(Spacer(1, 5))
    story.append(P("Training plots show loss at optimizer steps and validation diagnostics at epoch boundaries. With one full-model epoch, validation has one observation per run; it cannot establish a multi-epoch convergence trend. Response quality is assessed separately by blinded judgments."))

    story += [PageBreak(), section("4. DPO versus baseline results")]
    result_rows = [[P(x, True) for x in ("Set / model", "N", "W / T / L", "Unknown", "Win excl. ties", "Win half ties", "Retries")]]
    for name, label, n in (("hh", "HH DPO / SFT", 300), ("ultrafeedback", "UF DPO / SFT", 300), ("persona", "Persona LoRA / SmolLM2", 8)):
        s = summaries.get(name, {})
        result_rows.append([P(x, True) for x in (label, str(s.get("sample_count", n)),
            f"{s.get('wins','—')} / {s.get('ties','—')} / {s.get('losses','—')}",
            str(s.get("unknown", "—")), fmt(s.get("win_rate_excluding_ties")),
            fmt(s.get("win_rate_including_ties")), str(s.get("judge_case_retries", "—")))])
    table = Table(result_rows, colWidths=[1.45*inch, .35*inch, 1.0*inch, .65*inch, 1.0*inch, 1.0*inch, .55*inch], repeatRows=1)
    table.setStyle(TableStyle([("GRID", (0,0), (-1,-1), .3, colors.grey), ("BACKGROUND", (0,0), (-1,0), colors.lightgrey), ("VALIGN", (0,0), (-1,-1), "TOP")]))
    story += [table, Spacer(1, 7), P("Judge model: " + JUDGE_MODEL + ". Response order was randomized with seed 42; the judge prompt exposed only Response A/B. Persona judging also assessed energetic cooking-coach style alongside usefulness and safety. Requests used five pairs, at least five seconds between calls, three batch retries, then individual recovery. Unknowns and invalid judgments remain visible in the saved summaries."),
              P("Win excluding ties divides DPO wins by wins plus losses among resolved cases. Win with ties half gives each tie half credit. Persona is reported separately because its style target and same-base comparator differ from the full-model DPO tracks."),
              section("5. Interpretation")]
    for name, label in (("hh", "HH safety"), ("ultrafeedback", "AlpacaEval helpfulness"), ("persona", "Persona style")):
        s = summaries.get(name)
        if s and s.get("judged_count"):
            rate = s.get("win_rate_including_ties")
            direction = "favored the tuned model" if rate is not None and rate > .5 else "favored the baseline" if rate is not None and rate < .5 else "was evenly split"
            story.append(P(f"{label}: {s.get('wins')} wins, {s.get('ties')} ties, {s.get('losses')} losses among {s.get('judged_count')} resolved comparisons; half-tie rate {fmt(rate)} {direction}. This is a measured preference, not proof that the model is universally safer or more factual."))
        else:
            story.append(P(f"{label}: judging did not complete; no preference claim is made."))
    story.append(P("Interpretation: differences may reflect the match between each preference dataset and its evaluation prompts, a strong SFT baseline, single-seed training, and decoding sensitivity. The HH and UltraFeedback sets should not be pooled into one headline rate. The small persona set has especially high uncertainty."))

    story += [PageBreak(), section("6. Representative cases and error analysis")]
    for name, label in (("hh", "HH"), ("ultrafeedback", "UltraFeedback"), ("persona", "Persona")):
        pairs_path = OUT / "pairs" / f"{name}_blind.jsonl"
        judgments_path = OUT / "judgments" / f"{name}.jsonl"
        if not (pairs_path.is_file() and judgments_path.is_file()):
            continue
        pair_by_id = {str(row["id"]): row for row in read_records(pairs_path)}
        by_outcome = {"win": None, "loss": None, "tie": None}
        for judgment in read_records(judgments_path):
            pair = pair_by_id.get(str(judgment.get("id")))
            if not pair: continue
            winner = judgment.get("winner")
            outcome = "tie" if winner == "tie" else "win" if pair["metadata"].get(f"response_{winner}_model") == pair["metadata"]["dpo_model"] else "loss"
            if by_outcome[outcome] is None:
                by_outcome[outcome] = (pair, judgment)
        story.append(section(label))
        for outcome, selected in by_outcome.items():
            if selected is None: continue
            pair, judgment = selected
            story.append(P(f"{outcome.upper()} — Prompt: {pair['instruction'][:180]} | A: {pair['response_a'][:170]} | B: {pair['response_b'][:170]} | Judge reason: {judgment.get('reason','')[:180]}", tiny=True))
    persona_pairs_path = OUT / "pairs/persona_blind.jsonl"
    persona_judgments_path = OUT / "judgments/persona.jsonl"
    if persona_pairs_path.is_file() and persona_judgments_path.is_file():
        safety_pairs = {str(row["id"]): row for row in read_records(persona_pairs_path)
                        if str(row["id"]).endswith(("007", "008"))}
        safety_judgments = {str(row["id"]): row for row in read_records(persona_judgments_path)}
        story.append(section("Persona safety check"))
        for case_id, pair in safety_pairs.items():
            judgment = safety_judgments.get(case_id)
            if judgment:
                story.append(P(f"{case_id}: Judge chose {judgment.get('winner')}; reason: {judgment.get('reason','')[:240]}. Tuned response: {pair['response_a' if pair['metadata']['response_a_model'] == pair['metadata']['dpo_model'] else 'response_b'][:180]}", tiny=True))
            else:
                story.append(P(f"{case_id}: no valid safety judgment was returned.", tiny=True))
    story += [section("7. Limitations and reproducibility"),
              P("Model-proxy judging can be noisy and position-sensitive despite randomized order. Samples are fixed but small, with one training seed and one judge. Unknown judgments or proxy failures, if present, reduce the strength of the result. Persona has only eight prompts. The anti-repetition controls can change apparent quality; paired models use identical decoding."),
              P("Reproduction files: outputs/run_config.json, run_status.json, eval_sets/, sft/metrics.jsonl, dpo_hh/metrics.jsonl, dpo_ultrafeedback/metrics.jsonl, persona_lora/trainer_state.json, plots/, generations/, pairs/, judgments/, evaluation_summary.json, and the patched source files in PA2_evidence.zip. Best and final checkpoints are in the full saved Kaggle notebook output."),
              P("References: PA2 assignment manual; Qwen2.5-0.5B model card; Anthropic HH-RLHF dataset card; HuggingFaceH4 UltraFeedback Binarized dataset card; AlpacaEval; TRL and PEFT documentation.")]

    doc = SimpleDocTemplate(str(REPORT), pagesize=(8.5*inch, 11*inch),
                            leftMargin=.65*inch, rightMargin=.65*inch,
                            topMargin=.6*inch, bottomMargin=.6*inch)
    doc.build(story)
    from pypdf import PdfReader
    page_count = len(PdfReader(str(REPORT)).pages)
    print("Report:", REPORT, "pages:", page_count, "complete evaluation:", complete_eval)
    assert page_count <= 8, f"Report exceeds 8 pages: {page_count}"
    record("report", "complete" if complete_eval else "needs_review", f"{page_count} pages; full evaluation={complete_eval}")
except Exception as exc:
    record('report', 'failed', repr(exc))
    print('Report generation failed:', repr(exc))


In [ ]:
import zipfile
from pypdf import PdfReader

preflight = json.loads((OUT / "preflight.json").read_text(encoding="utf-8")) if (OUT / "preflight.json").is_file() else {}
def fixed_sets_ready():
    try:
        return all(len(read_records(OUT / name)) == expected for name, expected in (
            ("eval_sets/alpaca_eval_300_seed42.json", 300),
            ("eval_sets/hh_rlhf_eval_300_seed42.json", 300),
            ("eval_sets/persona_style_prompts.json", 8),
        ))
    except Exception:
        return False

checks = {
    "Two GPUs, public adapter checks, judge preflight": all(preflight.get(k) for k in
        ("two_gpus_visible", "public_adapter_tests_passed", "judge_ping_succeeded")),
    "SFT best and final": full_model(OUT / "sft/best") and full_model(OUT / "sft/final"),
    "HH DPO best and final": full_model(OUT / "dpo_hh/best") and full_model(OUT / "dpo_hh/final"),
    "UltraFeedback DPO best and final": full_model(OUT / "dpo_ultrafeedback/best") and full_model(OUT / "dpo_ultrafeedback/final"),
    "Persona LoRA adapter": lora_model(OUT / "persona_lora/best"),
    "Persona config and trainer history": all((OUT / p).is_file() for p in
        ("persona_lora/lora_config.json", "persona_lora/trainer_state.json")),
    "SFT and DPO metrics": all((OUT / p).is_file() for p in
        ("sft/metrics.jsonl", "dpo_hh/metrics.jsonl", "dpo_ultrafeedback/metrics.jsonl")),
    "Fixed evaluation sets": fixed_sets_ready(),
    "Base AlpacaEval generation": count_jsonl(OUT / "generations/base_alpaca.jsonl") == 300,
    "HH generations and pairs": all(count_jsonl(OUT / p) == 300 for p in ("generations/sft_hh.jsonl", "generations/dpo_hh_hh.jsonl", "pairs/hh_blind.jsonl")),
    "UltraFeedback generations and pairs": all(count_jsonl(OUT / p) == 300 for p in ("generations/sft_alpaca.jsonl", "generations/dpo_ultrafeedback_alpaca.jsonl", "pairs/ultrafeedback_blind.jsonl")),
    "Persona generations and pairs": all(count_jsonl(OUT / p) == 8 for p in ("generations/persona_base_style.jsonl", "generations/persona_lora_style.jsonl", "pairs/persona_blind.jsonl")),
    "Training plots": all((OUT / p).is_file() for p in ("plots/sft_loss.png", "plots/dpo_metrics.png", "plots/persona_loss.png")),
}
for name, expected in (("hh", 300), ("ultrafeedback", 300), ("persona", 8)):
    summary_path = OUT / "judgments" / f"{name}_summary.json"
    try:
        summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.is_file() else {}
    except Exception:
        summary = {}
    checks[f"{name} judgments complete"] = summary.get("judged_count") == expected and summary.get("unknown") == 0
REPORT = OUT / "report" / "PA2_final_report.pdf"
try:
    checks["Report PDF (8 pages maximum)"] = REPORT.is_file() and len(PdfReader(str(REPORT)).pages) <= 8
except Exception as exc:
    checks["Report PDF (8 pages maximum)"] = False
    print("Report PDF could not be read:", repr(exc))

final = {"all_complete": all(checks.values()), "checks": checks, "stage_status": STATUS,
         "repo_url": REPO_URL, "judge_model": JUDGE_MODEL}
(OUT / "final_checklist.json").write_text(json.dumps(final, indent=2), encoding="utf-8")

bundle = Path("/kaggle/working/PA2_evidence.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in OUT.rglob("*"):
        if not path.is_file(): continue
        relative = path.relative_to(OUT)
        if any(part in {"best", "final"} or part.startswith("checkpoint-") for part in relative.parts):
            continue
        if path.suffix in {".safetensors", ".bin", ".pt"}:
            continue
        archive.write(path, Path("outputs") / relative)
    for relative in ("submission_adapter.py", "requirements.txt", "README.md", "alignment/training.py", "scripts/build_pairwise.py", "scripts/plot_metrics.py", "scripts/train_persona_lora.py", "scripts/judge_pairs_reliable.py"):
        archive.write(REPO / relative, Path("source") / relative)

print("\nFINAL PA2 CHECKLIST")
for name, passed in checks.items():
    print(("PASS" if passed else "NEEDS ATTENTION"), name)
print("All complete:", final["all_complete"])
print("Report:", REPORT if REPORT.is_file() else "missing")
print("Evidence ZIP:", bundle, "size MiB:", round(bundle.stat().st_size / 2**20, 1))
print("Full model checkpoints and all output files will be in this saved Kaggle notebook version's Output section.")


## Submission after the saved run

Open the completed saved version's **Output** section and verify `outputs/final_checklist.json` says `all_complete: true`. Download `outputs/report/PA2_final_report.pdf`, the three training plots, `outputs/evaluation_summary.json`, and `PA2_evidence.zip`. Submit those with your GitHub repository URL. The full model checkpoints remain in the saved notebook output. If the checklist flags judge failures, the judgment files and failure logs are preserved so the same judging cell can be resumed in a new session after attaching this notebook output as an input.

A saved run cannot promise a particular grade: model-proxy quota, Kaggle GPU availability, and the grader's private tests are outside the notebook. The final checklist makes any incomplete requirement explicit.
